# SLM-KDN Colab Pro A100: RAG + Training + Testing

Use this notebook after syncing your repo to `master`. It clones the GitHub repo, installs dependencies, checks the A100, rebuilds the RAG index over `rag-doc/ex3300.pdf`, runs retrieval smoke tests, then runs preprocess, training, inference, and evaluation.

Before running: in Colab, choose **Runtime -> Change runtime type -> GPU**, then pick an A100 if available.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

## 2. Clone Your Repo from Master

In [ ]:
%cd /content
!rm -rf slm-kdn
!git clone --branch master https://github.com/Surya-Prasad/slm-kdn.git
%cd /content/slm-kdn
!git status --short
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

## 3. Install Dependencies

This installs the project requirements. `pypdf` is required for `rag-doc/ex3300.pdf` ingestion.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets pypdf

## 4. Hugging Face Login

Run this if your base model requires gated access. Use a token that has access to `meta-llama/Meta-Llama-3-8B`.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Confirm EX3300 Guide Is Present

In [ ]:
!find rag-doc -maxdepth 2 -type f -print
!ls -lh rag-doc/ex3300.pdf

## 6. Build/Test RAG Retrieval Only

This does not run training. It rebuilds `results/rag_index.pkl` and prints source file, page, score, and chunk preview.

In [ ]:
!python src/rag_query.py --rebuild --query "What are the front panel ports on a Juniper EX3300 switch?"
!python src/rag_query.py --query "How do I interpret ALM SYS MST LEDs on an EX3300?" --top_k 3
!python src/rag_query.py --query "What are the console port details for EX3300?" --top_k 3
!python src/rag_query.py --query "What power supply information does the EX3300 guide provide?" --top_k 3

## 7. Preprocess NIT Dataset

This preserves the original NIT pipeline and creates `data/processed/*.jsonl` files.

In [ ]:
!bash scripts/run_preprocess.sh
!find data/processed -maxdepth 1 -type f -print | sort
!head -n 1 data/processed/test.jsonl

## 8. Rebuild RAG After NIT Preprocessing

After preprocessing, the RAG index includes both the processed NIT JSONL records and the EX3300 guide.

In [ ]:
!python src/rag_query.py --rebuild --query "What are the front panel ports on a Juniper EX3300 switch?" --top_k 5

## 9. Train LoRA on A100

The repo config is already A100-oriented. If Colab runs out of memory, lower `training.max_seq_len` or batch settings in `config.yaml` before running this cell.

In [ ]:
!bash scripts/run_train.sh

## 10. RAG Inference Smoke Test

Create a tiny input file with EX3300-specific questions and run inference with retrieved EX3300 context. The `--rag_debug` output should show chunks from `ex3300.pdf`.

In [ ]:
import json, os
os.makedirs("data/processed", exist_ok=True)
rows = [
    {"intent": "What are the front panel ports on a Juniper EX3300 switch?", "context": "", "target_command": "", "category": "rag_ex3300"},
    {"intent": "How do I interpret ALM SYS MST LEDs on an EX3300?", "context": "", "target_command": "", "category": "rag_ex3300"},
    {"intent": "What are the console port details for EX3300?", "context": "", "target_command": "", "category": "rag_ex3300"},
]
with open("data/processed/rag_smoke.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

!python src/infer.py \
  --input_file data/processed/rag_smoke.jsonl \
  --output_file results/predictions/rag_smoke_predictions.jsonl \
  --use_rag \
  --rebuild_rag \
  --rag_debug

!cat results/predictions/rag_smoke_predictions.jsonl

## 11. Full Test Inference + Evaluation

Use this for the original NIT evaluation. Keep `--use_rag` on if you want retrieved context included during inference.

In [ ]:
!mkdir -p results/predictions results/metrics
!python src/infer.py \
  --input_file data/processed/test.jsonl \
  --output_file results/predictions/rag_predictions.jsonl \
  --use_rag \
  --rag_debug

!python src/evaluate.py --pred_file results/predictions/rag_predictions.jsonl
!cat results/metrics/eval_metrics.json

## 12. Save Results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/slm-kdn-results
!cp -r results /content/drive/MyDrive/slm-kdn-results/results-$(date +%Y%m%d-%H%M%S)
!ls -lh /content/drive/MyDrive/slm-kdn-results